In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import ipywidgets as widgets
from IPython.display import display
import trimesh
from torch import autograd as Grad
import random
import math
import json
from datetime import datetime

import implicit_reps
import importlib
importlib.reload(implicit_reps)
from implicit_reps import *

from bns_utils import *


device=torch.device('cpu')

In [2]:
'''
with open("configs/curves/circle5_jitter.json", "r") as f:
    config_dict = json.load(f)
'''

with open("configs/curves/shark15-normals.json", "r") as f:
    config_dict = json.load(f)



sdf_id = config_dict['shape_id']
coarse_edges_id = config_dict['coarse_edges_id']
overlap_param_v = config_dict['overlap_param_v']
if overlap_param_v<=0 or overlap_param_v>=1:
    print('Sorry, in the config we must have overlap param between 0 and 1.')
    
global_scale = config_dict['global_scale']
use_normals_loss = config_dict['use_normals_loss']

#blend_type = config_dict['blend_type']

blend_type = 'bary'
use_normals_loss=False


datestring = datetime.now().strftime("%y%m%d_%H%M")
run_id = coarse_edges_id + '_' + datestring

In [3]:

def local_parametrisation(t, mesh_path, n):

    tm = trimesh.load(mesh_path)
    V = torch.tensor(tm.vertices, dtype=torch.float)  # Ensure V is a tensor

    

    coarse_indices = [ int( j * (V.shape[0] - 1) / n) for j in range(n) ]
    
    coarse_points = V[coarse_indices,:2]

    out = torch.zeros(t.shape[0], n, 2)
    for j in range(n):
        if j==n-1:
            i = coarse_indices[n-1] + t*(V.shape[0] - coarse_indices[n-1])
        else:
            i = coarse_indices[j] + t*(coarse_indices[(j+1)%n] - coarse_indices[j])

        s, i = torch.frac(i), torch.floor(i).long()  # Use torch.frac() for fractional part
    
        out[:,j,:] = V[i, :2].squeeze() * (1 - s) + V[(i+1)%V.shape[0], :2].squeeze() * s


    return out, coarse_points


if blend_type=='default':
    
    def blend_func(x, v=overlap_param_v): #use the blend function from bns_utils
        result = B2_even(x, v=v)
        return result

elif blend_type=='simple_exp':

    def blend_func(x, v=overlap_param_v): #use the blend function from bns_utils
        result = B5_simple_exp(x, v=v)
        return result
        
elif blend_type=='bary':
    def blend_func(x, v=overlap_param_v): #use the blend function from bns_utils
        result = B0_linear(x, v=v)
        return result

def gradient( out, wrt, allow_unused=False):

		B = 1 if len(out.size()) < 3 else out.size(0)
		N = out.size(0) if len(out.size()) < 3 else out.size(1)
		R = out.size(-1)
		C = wrt.size(-1)

		gradients = []
		for dim in range(R):
			out_p = out[..., dim].flatten()

			select = torch.ones(out_p.size(), dtype=torch.float32).to(out.device)

			gradient = Grad.grad(outputs=out_p, inputs=wrt, grad_outputs=select, create_graph=True, allow_unused=allow_unused)[0]
			gradients.append(gradient)

		if len(out.size()) < 3:
			J_f_uv = torch.cat(gradients, dim=1).view(N, R, C)
		else:
			J_f_uv = torch.cat(gradients, dim=2).view(B, N, R, C)

		return J_f_uv




import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, layer_sizes, device=None):
        super(MLP, self).__init__()
        
        if device is None:
            device = torch.device('cpu')
        elif isinstance(device, str):
            device = torch.device(device)
        self.device = device

        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]).to(self.device))
            if i < len(layer_sizes) - 2:  # No activation on last layer
                layers.append(nn.Tanh())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # ensure input is on the same device
        if x.device != self.device:
            x = x.to(self.device)
        return self.network(x)



class BlendedMLP(nn.Module):
    def __init__(self, n, layer_sizes, coarse_points, global_scale, device):
        super(BlendedMLP, self).__init__()
        self.mlps = nn.ModuleList([MLP(layer_sizes) for _ in range(n)])
        self.coarse_points = coarse_points
        self.n = n
        self.weights = [1 for i in range(n)]
        self.global_scale = global_scale
        
        #self.rotations = [torch.eye(2) for i in range(n)]
        self.compute_fixed_rotations_and_scales()

        self.default_coarse_points = self.coarse_points
        self.default_rotations = self.rotations
        self.default_scales = self.scales

        
        self.fake_mlps = [lambda x: (torch.rand(1) + torch.sin(5*x)+2)*torch.stack([torch.cos(x), torch.sin(x)]).squeeze().transpose(0,1) for _ in range(n)]

    def reset(self):
        self.rotations = self.default_rotations
        self.scales = self.default_scales
        self.coarse_points = self.default_coarse_points
        self.weights = [1 for i in range(n)]

    def compute_fixed_rotations_and_scales(self):
        self.rotations = []
        self.scales = []
        for i in range(self.n):
            vec0 = self.coarse_points[i, :] - self.coarse_points[(i-1)%self.n, :]
            length0 = torch.sqrt((vec0*vec0).sum(-1))
            vec0 = vec0 / length0
            
            vec1 = self.coarse_points[(i+1)%self.n, :] - self.coarse_points[i, :]
            length1 = torch.sqrt((vec1*vec1).sum(-1))
            vec1 = vec1 / length1

            vec = (vec0+vec1)/2

            R = torch.tensor ([[vec[0], -1*vec[1] ],
                              [vec[1], vec[0] ] ]).transpose(1,0)

            self.rotations.append(R)

            s = self.global_scale * ( (length0+length1)/2.0 ) #scaling here helps the optimisation to not go wild

            self.scales.append(s)

    



    def save_mlps(self, filename="mlps.pth"):
        torch.save({
            'mlps_state_dict': [mlp.state_dict() for mlp in self.mlps],
        }, filename)
    
    def load_mlps(self, filename="mlps.pth"):
        checkpoint = torch.load(filename)
        
        for mlp, state_dict in zip(self.mlps, checkpoint['mlps_state_dict']):
            mlp.load_state_dict(state_dict)

    '''
    def reset_mlps(self):
        """Reinitialize all the MLP weights and biases to their default initialization."""
        for mlp in self.mlps:
            for layer in mlp.modules():
                if isinstance(layer, nn.Linear):
                    layer.reset_parameters()  # built-in PyTorch reinit. 
        print("All MLP weights and biases have been reset.")
    '''

    
    def reset_mlps(self):
        """Reinitialize all the MLP weights and biases using Xavier uniform initialization."""
        for mlp in self.mlps:
            for layer in mlp.modules():
                if isinstance(layer, nn.Linear):
                    # Xavier (Glorot) initialization for weights
                    torch.nn.init.xavier_uniform_(layer.weight)
    
                    # Bias initialized uniformly in the same range as Xavier recommends
                    if layer.bias is not None:
                        fan_in, fan_out = torch.nn.init._calculate_fan_in_and_fan_out(layer.weight)
                        bound = math.sqrt(6.0 / (fan_in + fan_out))
                        torch.nn.init.uniform_(layer.bias, -bound, bound)
    
        print("All MLP weights and biases have been reset using Xavier initialization.")
    
    
    '''
    def reset_mlps(self):
        """Set all MLP weights and biases to zero (deterministic reset)."""
        for mlp in self.mlps:
            for layer in mlp.modules():
                if isinstance(layer, nn.Linear):
                    # Zero all weights and biases
                    nn.init.constant_(layer.weight, 0.0)
                    if layer.bias is not None:
                        nn.init.constant_(layer.bias, 0.0)
        print("All MLP weights and biases have been set to zero.")
    '''

            
    def component(self, x, i, positioned=True):
        if positioned==True:
            return (self.weights[i] * self.scales[i] * self.mlps[i](x)@self.rotations[i]  + self.coarse_points[i] )
        else:
            return self.mlps[i](x)
        
       
    def forward(self, x):
        return torch.stack([ ( self.component(x,i)* blend_func(x) +
                              self.component(x+1,(i-1)%self.n)* blend_func(x+1) +
                              self.component(x-1, (i+1)%self.n )* blend_func(x-1) ) for i in range(self.n) ]).transpose(1,0)



    def diff_quants(self, x, i):
        x.requires_grad = True  # Enable differentiation
        r = self.forward(x)[:, i, :]  # Compute function output
        
        # Compute gradient of r w.r.t x
        tangent = gradient(out=r, wrt=x).squeeze()
    
    
        # Compute normalized tangent (unit vector)
        speed = torch.norm(tangent, dim=1, keepdim=True)
        
        unit_tangent = tangent / speed

        R = torch.tensor([[0.0,1.0,],
                          [-1.0,0.0]])
        
        unit_normal = unit_tangent @ R

        # Compute normal: derivative of tangent w.r.t x
        unit_tangent_deriv = gradient(out=unit_tangent, wrt=x).squeeze()

        print(unit_tangent_deriv.shape,  unit_normal.shape)

        norms = torch.norm(unit_tangent_deriv, dim=1, keepdim=True)

        k = torch.sign((unit_tangent_deriv * unit_normal).sum(-1)) * norms.squeeze() / speed.squeeze()

        
        return r, unit_tangent, unit_normal, k

        


In [4]:
'''
t = torch.arange(-2,2,0.001)
plt.plot(t, blend_func(t, v=0.5))
plt.plot(t, blend_func(t-1))
plt.plot(t, blend_func(t+1))
plt.savefig('blend_func.pdf')
plt.savefig('blend_func.png')
plt.show()
'''

"\nt = torch.arange(-2,2,0.001)\nplt.plot(t, blend_func(t, v=0.5))\nplt.plot(t, blend_func(t-1))\nplt.plot(t, blend_func(t+1))\nplt.savefig('blend_func.pdf')\nplt.savefig('blend_func.png')\nplt.show()\n"

In [5]:
def update_plot(**slider_values):
    import matplotlib.patches as patches

    t = torch.arange(-1, 1.01, 0.01, dtype=torch.float32).unsqueeze(-1)
    import matplotlib.pyplot as plt
    from matplotlib import gridspec
    
    fig = plt.figure(figsize=(16, 16))
    gs = gridspec.GridSpec(2, 2, )  
    #       ↑ width of col0 vs col1      ↑ height of row0 vs row1
    
    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])
    
    axes = [[ax00, ax01], [ax10, ax11]]

    
    # Extract weights from slider values
    blended_mlp.weights = [slider_values[f'weight_{i}'] for i in range(blended_mlp.n)]
    current_t = torch.tensor(slider_values['t slider'], dtype=torch.float32).unsqueeze(0)

    # Compute components
    components = [blended_mlp.component(t, i) for i in range(blended_mlp.n)]
    unpositioned_components = [blended_mlp.component(t, i, positioned=False) for i in range(blended_mlp.n)]

    output = blended_mlp(0.5 * t)
    current_output = blended_mlp(current_t).transpose(0, 1)
    current_component_part = [blended_mlp.component(current_t, i) for i in range(blended_mlp.n)]

    target, _ = local_parametrisation(torch.arange(0, 1.0, 0.01).unsqueeze(-1),
                                      mesh_path=mesh_path, n=n)

    colors = [plt.get_cmap("tab10")(i % 10) for i in range(blended_mlp.n)]

    # -------------------------------------------------------
    # [0,1] — COMPONENTS at Coarse Points
    # -------------------------------------------------------

    axes[0] [1].set_xlim([-1.2, 1.2])
    axes[0] [1].set_ylim([-1.2, 1.2])
    axes[0] [1].set_aspect('equal')
    axes[0] [1].set_title("Components at Coarse Points")

    scale = 0.1 ### just for illustration purposes
    
    for i in range(blended_mlp.n):
        axes[0] [1].plot(target.detach()[:, i, 0], target.detach()[:, i, 1],
                        color='red', alpha=0.2, linestyle=':')
        color = colors[i]
        center = blended_mlp.coarse_points[i].detach().numpy()
        comp = unpositioned_components[i].detach().numpy()

        # Scale and translate to coarse vertex
        mini = comp * scale + center

        # Draw scaled-down component
        axes[0] [1].plot(mini[:, 0], mini[:, 1], color=color, linewidth=1)
        axes[0] [1].scatter(mini[::5, 0], mini[::5, 1],
                           color='black', marker='o', s=2.0)

        # Bounding box
        xmin, xmax = mini[:, 0].min(), mini[:, 0].max()
        ymin, ymax = mini[:, 1].min(), mini[:, 1].max()
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                 linewidth=0.5, edgecolor='black', facecolor='none', alpha=0.6)
        axes[0] [1].add_patch(rect)

        # Mark coarse vertex
        axes[0] [1].scatter(center[0], center[1], color=color, marker='x', s=20, alpha=0.7)

    # -------------------------------------------------------
    # [0,0] — PARAMETER INTERVAL THUMBNAILS
    # -------------------------------------------------------
    '''
    axes[0] [0].set_title("Parameter Intervals [-1, 1]")
    axes[0] [0].set_xlim([-1.1, 1.1])
    axes[0] [0].set_ylim([-0.5, blended_mlp.n - 0.5])
    axes[0] [0].set_aspect('auto')

    for i in range(blended_mlp.n):
        color = colors[i]
        y = np.full_like(t.squeeze().numpy(), i)  # horizontal line position
        x = t.squeeze().numpy()

        # Draw interval line
        axes[0] [0].plot(x, y, color=color, linewidth=1.5)

        # Sparse black dots
        step = 5
        axes[0] [0].scatter(x[::step], y[::step],
                           color='black', s=8, alpha=0.8, zorder=3)

        # Label component index
        axes[0] [0].text(1.05, i, f"W{i}", va='center', ha='left', color=color)

    axes[0] [0].set_yticks(range(blended_mlp.n))
    axes[0] [0].set_yticklabels([f"Comp {i}" for i in range(blended_mlp.n)])
    axes[0] [0].grid(True, alpha=0.2)
    '''

    # -------------------------------------------------------
    # [1,0] — COMPONENTS in canonical space
    # -------------------------------------------------------
    axes[1] [0].set_xlim([-1.2, 1.2])
    axes[1] [0].set_ylim([-1.2, 1.2])
    axes[1] [0].set_aspect('equal')
    axes[1] [0].set_title("Unblended Components (Rotated and Scaled)")

    for i in range(blended_mlp.n):
        color = colors[i]
        axes[1] [0].plot(components[i][:, 0].detach(), components[i][:, 1].detach(),
                        linewidth=1, alpha=0.9, label=str(i), color=color)
        axes[1] [0].scatter(components[i][::5, 0].detach(), components[i][::5, 1].detach(),
                           color='black', marker='o', s=2.0)
        axes[1] [0].scatter(current_component_part[i][0].detach(),
                           current_component_part[i][1].detach(),
                           linewidth=1, alpha=0.5, s=10, color=color)

    for i in range(blended_mlp.n):
        for k in range(components[i].shape[0] // 10):
            halfway = components[i].shape[0] // 2
            axes[1] [0].plot([components[i][5*k + halfway, 0].detach(),
                             components[(i + 1) % n][5*k, 0].detach()],
                            [components[i][5*k + halfway, 1].detach(),
                             components[(i + 1) % n][5*k, 1].detach()],
                            color='black', linestyle='-', alpha=0.2)

    ###  axes[1] [0].legend()

    # -------------------------------------------------------
    # [1,1] — COMBINED SHAPE
    # -------------------------------------------------------
    axes[1] [1].set_title("Blended Curve")
    axes[1] [1].set_xlim([-1.2, 1.2])
    axes[1] [1].set_ylim([-1.2, 1.2])
    axes[1] [1].set_aspect('equal', adjustable='box')

    for i in range(blended_mlp.n):
        color = colors[i]
        axes[1] [1].plot(output.detach()[:, i, 0], output.detach()[:, i, 1],
                        linewidth=1, alpha=1, linestyle='-', color=color, label=str(i))
        axes[1] [1].scatter(blended_mlp.coarse_points[i, 0], blended_mlp.coarse_points[i, 1],
                           alpha=0.5, color=color, marker='x')
        axes[1] [1].scatter(current_output.detach()[i, 0], current_output.detach()[i, 1],
                           color=color, alpha=0.5)
        #axes[1] [1].scatter(output.detach()[::5, i, 0], output.detach()[::5, i, 1],
        #                   color='black', marker='o', s=2.0)
    ### axes[1] [1].legend()

    # -------------------------------------------------------
    # Layout cleanup
    # -------------------------------------------------------
    #for row in axes:
    #    for ax in row:
    #        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    if hasattr(update_plot, "_save_filename") and update_plot._save_filename:
        fig.savefig(update_plot._save_filename, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()



def show(save=None):

    update_plot._save_filename = save  # tell update_plot whether to save

    if save:
        # Run once without creating sliders
        slider_values = {f'weight_{i}': 1.0 for i in range(blended_mlp.n)}
        slider_values['t slider'] = 0.0
        update_plot(**slider_values)
    else:
        # Normal interactive mode
        sliders = {
            f'weight_{i}': widgets.FloatSlider(min=0, max=1, step=0.1, value=1, description=f"W{i}")
            for i in range(blended_mlp.n)
        }
        sliders['t slider'] = widgets.FloatSlider(min=-1, max=1, step=0.01, value=0, description='t slider')

        interactive_plot = widgets.interactive(update_plot, **sliders)
        display(interactive_plot)




def update_diff_quant_plot(t_slider, r, unit_tangent, unit_normal, k, all_t):
    t = t_slider

    # Convert PyTorch tensors to NumPy (if they are not already)
    r_np = r.detach().cpu().numpy()
    unit_normal_np = unit_normal.detach().cpu().numpy()

    # Create figure and axis
    fig, ax = plt.subplots()

    # Extract components for quiver plot
    U = unit_normal_np[::10, 0]  # x-component
    V = unit_normal_np[::10, 1]  # y-component

    # Plot normals using quiver
    ax.quiver(
        r_np[::10, 0], r_np[::10, 1], U, V, 
        angles='xy', scale_units='xy', scale=50, 
        color='b', alpha=0.2, label='Normals',
        headwidth=0, headlength=0, headaxislength=0
    )

    ax.set_aspect('equal')

    # Plot the main curve
    ax.plot(r_np[:, 0], r_np[:, 1], color='black', linewidth=0.5, alpha=0.2)




    radii = 1/torch.abs(k).detach()
    circle_centres = r.detach() + torch.sign(k).unsqueeze(-1) * radii.unsqueeze(-1) * unit_normal.detach() 

    #print('all t shape', all_t.shape)
    t_index = ((t+1)/2) * (all_t.shape[0]-1)

    #print(t_index, int(t_index))
    t_index=round(t_index)

    #print(all_t[t_index,:])
    radius = radii[t_index]
    (x,y) = circle_centres[t_index,:]

    xlim = ax.get_xlim()
    ylim=ax.get_ylim()

    circle = patches.Circle((x, y), radius, fill=False, linewidth=1, alpha=0.8, color='black')
    ax.scatter(r.detach()[t_index,0], r.detach()[t_index,1], marker='x', color='black')
    ax.add_patch(circle)  # Use ax.add_patch() instead of plt.add_patch()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


    

    plt.show()

def show_diff_quant(i=0):
    # Create slider widget
    t_slider = widgets.FloatSlider(min=-1, max=1, step=0.001, value=0, description='t slider')

    # Generate differential quantities (assuming `blended_mlp.diff_quants` is correct)
    all_t = torch.arange(-1, 1.001, 0.001).unsqueeze(-1)

    #print(all_t)
    r, unit_tangent, unit_normal, k = blended_mlp.diff_quants(all_t, i=i)


    # Create interactive plot
    interactive_plot = widgets.interactive(
        update_diff_quant_plot, 
        t_slider=t_slider, 
        r=widgets.fixed(r), 
        unit_tangent=widgets.fixed(unit_tangent), 
        unit_normal=widgets.fixed(unit_normal), 
        k=widgets.fixed(k),
        all_t=widgets.fixed(all_t)
    )

    display(interactive_plot)





def update_deform_plot( **slider_values ):
    t =  torch.arange(-1, 1.01, 0.01).unsqueeze(-1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Extract weights from slider values
    #blended_mlp.weights = [slider_values[f'weight_{i}'] for i in range(blended_mlp.n)]
    pos = torch.stack([ torch.tensor([slider_values[f'x_disp_{i}'], slider_values[f'y_disp_{i}'] ]) 
                       for i in range(blended_mlp.n)])
    
    blended_mlp.coarse_points = pos
    blended_mlp.compute_fixed_rotations_and_scales()

    


    components = [blended_mlp.component(t,i) for i in range(blended_mlp.n)]
    
    output = blended_mlp(0.5*t)
    

    #print('cur output', current_output.shape)

    #blend_weight = blend_func(t)
    target, _ = local_parametrisation(torch.arange(0,1.0, 0.01).unsqueeze(-1), mesh_path = mesh_path, n=n)
    
    #print(output.shape)
   
    
    axes[0].set_xlim([-1.2, 1.2])
    axes[0].set_ylim([-1.2, 1.2])
    axes[0].set_title("Components")

    colors = [plt.get_cmap("tab10")(i % 10) for i in range(blended_mlp.n)]
    for i in range(blended_mlp.n):
        #output = torch.tensor(output)
        #print(output)
        color=colors[i]
        axes[0].plot(components[i][:,0].detach(), components[i][:,1].detach(), linewidth=1, alpha=0.9, label=str(i), color=color)

        
        
        axes[0].plot(output.detach()[:,i, 0], output.detach()[:,i, 1], color='blue', linewidth=1, alpha=0.1)


        axes[1].plot(output.detach()[:, i, 0], output.detach()[ :,i, 1], linewidth=1, alpha=1, linestyle='-', color=color, label=str(i))
        axes[1].plot(target.detach()[:,i,0], target.detach()[:,i,1], color='red', alpha=0.2, linestyle=':')
    
        axes[1].scatter(blended_mlp.coarse_points[i,0], blended_mlp.coarse_points[i,1], alpha=0.5, color=color, marker='x')

        
    axes[0].plot([],[], color='blue', linewidth=1, alpha=0.1, label='combined')

        
    axes[0].legend()
    axes[1].legend()


    
    axes[1].set_title("Combined Shape")
    axes[1].set_xlim([-1.2, 1.2])
    axes[1].set_ylim([-1.2, 1.2])

    plt.show()


def show_deform():
   
    sliders = {
        **{f'x_disp_{i}': widgets.FloatSlider(min=-1, max=1, step=0.001, value=blended_mlp.coarse_points[i,0], description=f"x_disp_{i}") for i in range(blended_mlp.n)},
        **{f'y_disp_{i}': widgets.FloatSlider(min=-1, max=1, step=0.001, value=blended_mlp.coarse_points[i,1], description=f"y_disp_{i}") for i in range(blended_mlp.n)}
    }

    
    # Create interactive widget with all sliders
    interactive_plot = widgets.interactive(update_deform_plot, **sliders)
    
    # Display sliders and plot
    display(interactive_plot)





















In [6]:

layer_sizes = [1, 16, 16, 2]
mesh_path = 'data/curves/'+coarse_edges_id+'.obj'



mesh = trimesh.load(mesh_path)

coarse_points = torch.tensor( mesh.vertices[:,:2], dtype=torch.float32 )
n = coarse_points.shape[0]

blended_mlp = BlendedMLP(n, layer_sizes, coarse_points, global_scale, device)

blended_mlp = blended_mlp.to(device)



In [7]:

show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [8]:
epoch=0
blended_mlp.reset_mlps()

All MLP weights and biases have been reset using Xavier initialization.


In [9]:
import os

os.makedirs(f"output/curves/{run_id}", exist_ok=True)

optimizer = optim.Adam(blended_mlp.parameters(), lr=0.01)
plot_normals=False



curve_model = load_deepsdf_curve_model(
    weights_path="sdf_weights/curves/shark.pth",
    device=device
)



# Training loop

for _ in range(100000):
    optimizer.zero_grad()

    current_t = torch.arange(0.001, 0.999, 0.01, device=device).unsqueeze(-1)
    current_t.requires_grad = True
    
    output = blended_mlp(current_t)

    #print('out shape', output.shape)

    cur_sdf = sdf(output, sdf_id=sdf_id, model=curve_model, squared=False)
    cur_squared_sdf = sdf(output, sdf_id=sdf_id, model=curve_model, squared=True) ### prevent NaNs when differentiating by avoiding unsquared sdf.

    sdf_loss = (cur_squared_sdf).mean()

    use_normals_loss=True
    if use_normals_loss == True:

        sdf_gradient = gradient(out = cur_sdf, wrt = output).squeeze()

        all_unit_normals = torch.zeros(output.shape)
        
        for i in range(output.shape[1]):
    
            tangent = gradient(out=output[:,i,:], wrt=current_t).squeeze()
            
            # Compute normalized tangent (unit vector)
            speed = torch.norm(tangent, dim=1, keepdim=True)
                
            unit_tangent = tangent / speed
        
            R = torch.tensor([[0.0,1.0,],
                                  [-1.0,0.0]])
        
            unit_normal = - unit_tangent @ R    #### sign correction to make it match the sdf-gradient
    
            all_unit_normals[:,i,:] = unit_normal
        
        #print(all_unit_normals.shape, sdf_gradient.shape)  
    
        normals_loss = (- (sdf_gradient * all_unit_normals).sum(-1) + 1.0).mean()

    

        loss = sdf_loss + 0.1 * normals_loss

    else:
        
        loss = sdf_loss


    
    loss.backward()
    optimizer.step()

    # Corrected indentation for if condition
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    
    if epoch % 1000 == 0:
        # Optional: save current plot using your existing show()
        show(save=f'output/curves/{run_id}/plot_{epoch:010d}.png')
        show(save=f'output/curves/{run_id}/plot_{epoch:010d}.pdf')


        if use_normals_loss ==True:

            # Flatten batch and component dimensions to 2D arrays for plotting
            pts = output.detach().reshape(-1, 2).numpy()
            normals = all_unit_normals.detach().reshape(-1, 2).numpy()

            if plot_normals==True:
    
                # --- Minimal quiver plot for debugging ---
                plt.figure(figsize=(12,12))
                
                
            
                # Plot the points
                plt.scatter(pts[:,0], pts[:,1], s=1, color='blue', alpha=0.9)
            
                # Plot the normals
                plt.quiver(pts[:,0], pts[:,1], normals[:,0], normals[:,1], color='red', alpha=0.5, angles='xy', scale_units='xy', scale=10.0)
            
                plt.axis('equal')
                plt.title(f"Unit normals (epoch {epoch})")
                plt.show()
    
    epoch += 1  # Increment epoch counter







Loading 2D curve SDF model onto cpu...
2D Curve SDF Model loaded successfully on device: cpu


/Users/romywilliamson/Documents/BNS/bns/sdf_fitter_2d_utils.py:135: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(points, dtype=torch.float32, device=device)
/Users/romywilliamson/Documents/BNS/bns/sdf_fitter_2d_utils.py:136: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(sdf_vals, dtype=torch.float32, device=device).unsqueeze(-1)
/Users/romywilliamson/Documents/BNS/bns/implicit_reps.py:606: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.

Epoch 0, Loss: 0.031751
Epoch 10, Loss: 0.022898
Epoch 20, Loss: 0.020993
Epoch 30, Loss: 0.021254
Epoch 40, Loss: 0.021869
Epoch 50, Loss: 0.020456
Epoch 60, Loss: 0.022574
Epoch 70, Loss: 0.022151
Epoch 80, Loss: 0.020609
Epoch 90, Loss: 0.017491
Epoch 100, Loss: 0.018733
Epoch 110, Loss: 0.019533
Epoch 120, Loss: 0.019260
Epoch 130, Loss: 0.018974
Epoch 140, Loss: 0.018772


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10783bdc0>>
Traceback (most recent call last):
  File "/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 

KeyboardInterrupt



In [10]:
# Print the device of all parameters
for name, param in curve_model.named_parameters():
    print(name, param.device)

0.weight cpu
0.bias cpu
2.weight cpu
2.bias cpu
4.weight cpu
4.bias cpu
6.weight cpu
6.bias cpu
8.weight cpu
8.bias cpu


In [11]:
show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [12]:
device

device(type='cpu')

In [13]:
import imageio.v2 as imageio
import os
import numpy as np

def make_gif(folder, output_path='output.gif', fps=10, end_pause_frames=10):
    # Get all image files in order
    files = sorted(
        [f for f in os.listdir(folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    )

    if not files:
        print("No image files found in folder!")
        return

    # Read first image to get target size
    first_img = imageio.imread(os.path.join(folder, files[0]))
    target_h, target_w = first_img.shape[:2]

    images = []
    for f in files:
        img = imageio.imread(os.path.join(folder, f))
        h, w = img.shape[:2]

        # Crop if larger than target size
        if h > target_h:
            img = img[:target_h, :]
        if w > target_w:
            img = img[:, :target_w]

        # Optionally pad if smaller (fill with black)
        if h < target_h or w < target_w:
            new_img = np.zeros((target_h, target_w, img.shape[2]), dtype=img.dtype)
            new_img[:h, :w] = img
            img = new_img

        images.append(img)

    # Add blank frames at the end
    blank = np.zeros_like(images[0])
    images.extend([blank] * end_pause_frames)

    # Write GIF
    imageio.mimsave(output_path, images, fps=fps)
    print(f"✅ Saved GIF to {output_path}")

# Example usage

make_gif('output/curves/' + run_id, 
         'output/curves/' + run_id + '/animation_'+run_id+'.gif', 
         fps=5)


✅ Saved GIF to output/curves/shark_bdry_coarse15_260401_1430/animation_shark_bdry_coarse15_260401_1430.gif


In [14]:
blended_mlp.reset()
show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [15]:
blended_mlp.save_mlps('models/curves/'+run_id+'.pth')
#blended_mlp.load_mlps('models/curves/circle5.pth')

In [16]:
blended_mlp.reset()
show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [17]:
############# Rotation Equivariance ############
blended_mlp.reset()
R = torch.tensor([[0.0, 1.0],
                  [-1.0, 0.0]])

blended_mlp.coarse_points = blended_mlp.coarse_points @ R

blended_mlp.compute_fixed_rotations_and_scales()


show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [18]:
############# Translation Equivariance ############
blended_mlp.reset()
T = torch.tensor([-0.5, +0.3 ])

blended_mlp.coarse_points = blended_mlp.coarse_points + T

blended_mlp.compute_fixed_rotations_and_scales()

show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [19]:
############# Scale Equivariance ############
blended_mlp.reset()
s = 0.5

blended_mlp.coarse_points = blended_mlp.coarse_points * s

blended_mlp.compute_fixed_rotations_and_scales()


show()

interactive(children=(FloatSlider(value=1.0, description='W0', max=1.0), FloatSlider(value=1.0, description='W…

In [ ]:
############# Translating a Single Coarse Vertex ############
blended_mlp.reset()
#T = torch.zeros_like(blended_mlp.coarse_points)
#T[0,:] = torch.tensor([-0.1, +0.1 ])

#blended_mlp.coarse_points = blended_mlp.coarse_points + T

blended_mlp.compute_fixed_rotations_and_scales()

show()

In [ ]:
blended_mlp.reset()
show_deform()

In [ ]:
for i in range(blended_mlp.n):
    show_diff_quant(i=i);

In [ ]:
torch.__version__

In [ ]:
show_diff_quant(i=6)